# IMDB_Sentiment_Analysis_Project

## Import Libraries

In [13]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer


In [14]:
# Load dataset
df = pd.read_csv("IMDB Dataset.csv.zip")

# Check first 5 rows
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [15]:

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\anjal\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\anjal\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\anjal\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\anjal\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

## #text-preprocessing

In [16]:
def preprocess_text(text):
    
    # 1. Lowercase
    text = text.lower()
    
    # 2. Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    
    # 3. Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    
    # 4. Remove punctuation & numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # 5. Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    # 6. Tokenization
    tokens = word_tokenize(text)
    
    # 7. Stopword removal
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]
    
    # 8. Lemmatization
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    # 9. Join back to sentence
    clean_text = " ".join(tokens)
    
    return clean_text

In [18]:
df['clean_review'] = df['review'].apply(preprocess_text)
df[['review', 'clean_review']].head()

,review,clean_review
0,One of the other reviewers has mentioned that ...,one reviewer mentioned watching oz episode you...
1,A wonderful little production. <br /><br />The...,wonderful little production filming technique ...
2,I thought this was a wonderful way to spend ti...,thought wonderful way spend time hot summer we...
3,Basically there's a family where a little boy ...,basically there family little boy jake think t...
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter matteis love time money visually stunni...


### Total Words and Unique Words

In [7]:
all_text = " ".join(df["clean_review"])
words = all_text.split()
total_words = len(words)
unique_words = len(set(words))

print("Total words:", total_words)
print("Unique words (Vocabulary size):", unique_words)

Total words: 5928792
Unique words (Vocabulary size): 203244


## #Feature Extraction

### One Hot Encoding

In [8]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder()

sentiment_encoded = encoder.fit_transform(df[["sentiment"]])

print(sentiment_encoded.toarray())

[[0. 1.]
 [0. 1.]
 [0. 1.]
 ...
 [1. 0.]
 [1. 0.]
 [1. 0.]]


### Bag of Words

In [9]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer()
bow = cv.fit_transform(df["clean_review"])
vocabulary = cv.get_feature_names_out()


print("Vocabulary size:", len(vocabulary))
print("First 20 words:", vocabulary[:20])

Vocabulary size: 203220
First 20 words: ['aa' 'aaa' 'aaaa' 'aaaaaaaaaaaahhhhhhhhhhhhhh' 'aaaaaaaargh' 'aaaaaaah'
 'aaaaaaahhhhhhggg' 'aaaaagh' 'aaaaah' 'aaaaargh'
 'aaaaarrrrrrgggggghhhhhh' 'aaaaatchkah' 'aaaaaw' 'aaaahhhhhh'
 'aaaahhhhhhh' 'aaaand' 'aaaarrgh' 'aaaggghhhhhhh' 'aaaghi' 'aaah']


### Word frequency

In [10]:
import numpy as np
word_freq = bow.sum(axis=0)
word_freq = np.asarray(word_freq).ravel()
freq_df = pd.DataFrame({
    "word": vocabulary,
    "count": word_freq
})

freq_df.sort_values(by="count", ascending=False).head(10)

,word,count
116689,movie,99025
63274,film,89809
126567,one,52677
102271,like,39790
180767,time,29396
73439,good,28615
28803,character,27573
71525,get,24435
57409,even,24286
170086,story,24229


### Bigram and Trigram

In [11]:
# Bigram

bigram = CountVectorizer(ngram_range=(2,2))
bigram_matrix = bigram.fit_transform(df["clean_review"])
vocabulary = bigram.get_feature_names_out()
print("Bigram vocabulary size:", len(bigram.get_feature_names_out()))
print("First 20 words:", vocabulary[:20])

# Trigram

trigram = CountVectorizer(ngram_range=(3,3))
trigram_matrix = trigram.fit_transform(df["clean_review"])

print("Trigram vocabulary size:", len(trigram.get_feature_names_out()))

Bigram vocabulary size: 3069985
First 20 words: ['aa acting' 'aa antic' 'aa cultrehab' 'aa date' 'aa doctor' 'aa group'
 'aa jaega' 'aa level' 'aa meeting' 'aa meri' 'aa milne' 'aa mindless'
 'aa presentation' 'aa rating' 'aa uo' 'aa violent' 'aa yet' 'aaa ball'
 'aaa even' 'aaa famous']
Trigram vocabulary size: 5369751


### TF-IDF

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(df["clean_review"])
vocab = tfidf.get_feature_names_out()
idf_scores = tfidf.idf_

tfidf_df = pd.DataFrame({
    "word": vocab,
    "idf_score": idf_scores
})

tfidf_df.head()

,word,idf_score
0,aa,8.986585
1,aaa,9.254849
2,aaaa,11.126651
3,aaaaaaaaaaaahhhhhhhhhhhhhh,11.126651
4,aaaaaaaargh,11.126651
